In [ ]:
!pip -q install minerva-ml

In [2]:
from pathlib import Path
from PIL import Image
import copy
import cv2
import csv
import random
import numpy as np
from minerva.data.readers.png_reader import PNGReader
from minerva.data.readers.csv_reader import CSVReader
#from minerva.data.readers.patched_array_reader import NumpyArrayReader
from minerva.data.data_modules.base import MinervaDataModule

from minerva.transforms.transform import _Transform, Identity
from minerva.transforms.random_transform import _RandomSyncedTransform, RandomFlip, RandomGrayScale, RandomSolarize, RandomRotation
from minerva.data.datasets.base import SimpleDataset

#from minerva.analysis.model_analysis import TSNEAnalysis
#from minerva.analysis.metrics.pixel_accuracy import PixelAccuracy

from minerva.utils.tensor import to_tensor

#from minerva.models.ssl.byol import BYOL

from torchvision import transforms
import lightning as L
import torch
from torchmetrics import JaccardIndex
import matplotlib.pyplot as plt
import torchvision.models as models
from torch.utils.data import DataLoader
from torchmetrics.classification import MulticlassJaccardIndex
import torchvision.transforms.functional as F
#from torchvision.models import resnet50
import torch.nn as nn
from torchmetrics.classification import Accuracy
from torchvision.models.resnet import resnet50
from lightning.pytorch.callbacks import Callback, ModelCheckpoint
from torchvision.models._utils import IntermediateLayerGetter
from torchvision.models.segmentation import deeplabv3_resnet50

from sklearn.manifold import TSNE
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

import zipfile
import os

In [3]:
from google.colab import drive
drive.mount('/content/drive')
DIR = '/content/drive/My Drive/iniciacao_cientifica/'

Mounted at /content/drive


# deeplab code

In [5]:
from typing import Any, Dict, Optional, Union, Callable

import lightning as L
import torch
from torchmetrics import Metric
from minerva.models.loaders import LoadableModule


class SimpleSupervisedModel(L.LightningModule):
    """A modular Lightning model wrapper for supervised learning tasks.

    This class enables the construction of supervised models by combining a
    backbone (feature extractor), an optional adapter, and a fully connected
    (FC) head. It provides a clean interface for setting up custom training,
    validation, and testing pipelines with pluggable loss functions, metrics,
    optimizers, and learning rate schedulers.

    The architecture is structured as follows:

        +------------------+
        |  Backbone Model  |
        +------------------+
                |
                v
        +------------------------+
        |  Adapter (Optional)    |
        +------------------------+
                |
         (Flatten if needed)
                v
        +------------------------+
        |  Fully Connected Head  |
        +------------------------+
                |
                v
        +------------------+
        |  Loss Function   |
        +------------------+

    Training and validation steps comprise the following steps:

    1. Forward pass input through the backbone.
    2. Pass through adapter (if provided).
    3. Flatten the output (if `flatten` is True) before the FC head.
    4. Forward through the FC head.
    5. Compute loss with respect to targets.
    6. Backpropagate and update parameters.
    7. Compute metrics and log them.
    8. Return loss. `train_loss`, `val_loss`, and `test_loss` are always
       logged, along with any additional metrics specified in the
       `train_metrics`, `val_metrics`, and `test_metrics` dictionaries.

    This wrapper is especially useful to quickly set up supervised models for
    various tasks, such as image classification, object detection, and
    segmentation. It is designed to be flexible and extensible, allowing users
    to easily swap out components like the backbone, adapter, and FC head as
    needed. The model is built with a focus on simplicity and modularity, making
    it easy to adapt to different use cases and requirements.
    The model is designed to be used with PyTorch Lightning and is compatible
    with its training loop.

    **Note**: For more complex architectures that does not follow the above
    structure should not inherit from this class.

    **Note**: Input batches must be tuples (input_tensor, target_tensor).
    """

    def __init__(
        self,
        backbone: Union[torch.nn.Module, LoadableModule],
        fc: Union[torch.nn.Module, LoadableModule],
        loss_fn: torch.nn.Module,
        adapter: Optional[Callable[[torch.Tensor], torch.Tensor]] = None,
        learning_rate: float = 1e-3,
        flatten: bool = True,
        train_metrics: Optional[Dict[str, Metric]] = None,
        val_metrics: Optional[Dict[str, Metric]] = None,
        test_metrics: Optional[Dict[str, Metric]] = None,
        #freeze_backbone: bool = False,
        freeze_backbone: bool = True,
        optimizer: type = torch.optim.Adam,
        optimizer_kwargs: Optional[Dict[str, Any]] = None,
        lr_scheduler: Optional[type] = None,
        lr_scheduler_kwargs: Optional[Dict[str, Any]] = None,
    ):
        """
        Initializes the supervised model with training components and configs.

        Parameters
        ----------
        backbone : torch.nn.Module or LoadableModule
            The backbone (feature extractor) model.
        fc : torch.nn.Module or LoadableModule
            The fully connected head. Use nn.Identity() if not required.
        loss_fn : torch.nn.Module
            Loss function to optimize during training.
        adapter : Callable, optional
            Function to transform backbone outputs before feeding into `fc`.
        learning_rate : float, default=1e-3
            Learning rate used for optimization.
        flatten : bool, default=True
            If True, flattens backbone outputs before `fc`.
        train_metrics : dict, optional
            TorchMetrics dictionary for training evaluation.
        val_metrics : dict, optional
            TorchMetrics dictionary for validation evaluation.
        test_metrics : dict, optional
            TorchMetrics dictionary for test evaluation.
        freeze_backbone : bool, default=False
            If True, backbone parameters are frozen during training.
        optimizer: type
            Optimizer class to be instantiated. By default, it is set to
            `torch.optim.Adam`. Should be a subclass of
            `torch.optim.Optimizer` (e.g., `torch.optim.SGD`).
        optimizer_kwargs : dict, optional
            Additional kwargs passed to the optimizer constructor.
        lr_scheduler : type, optional
            Learning rate scheduler class to be instantiated. By default, it is
            set to None, which means no scheduler will be used. Should be a
            subclass of `torch.optim.lr_scheduler.LRScheduler` (e.g.,
            `torch.optim.lr_scheduler.StepLR`).
        lr_scheduler_kwargs : dict, optional
            Additional kwargs passed to the scheduler constructor.
        """

        super().__init__()
        self.backbone = backbone
        self.fc = fc
        self.loss_fn = loss_fn
        self.adapter = adapter
        self.learning_rate = learning_rate
        self.flatten = flatten
        self.freeze_backbone = freeze_backbone
        self.metrics = {
            "train": train_metrics,
            "val": val_metrics,
            "test": test_metrics,
        }
        self.optimizer = optimizer
        self.optimizer_kwargs = optimizer_kwargs or {}
        self.lr_scheduler = lr_scheduler
        self.lr_scheduler_kwargs = lr_scheduler_kwargs or {}
        # add train and val loss list
        self.train_losses = []
        self.val_losses = []
        self.train_accuracies = []
        self.val_accuracies = []
        self.train_loss_batches = []
        self.val_loss_batches = []

    def on_train_epoch_end(self):
        if self.train_loss_batches:
            avg = torch.stack(self.train_loss_batches).mean().item()
            self.train_losses.append(avg)
            self.train_loss_batches.clear()

    def on_validation_epoch_end(self):
        if self.val_loss_batches:
            avg = torch.stack(self.val_loss_batches).mean().item()
            self.val_losses.append(avg)
            self.val_loss_batches.clear()

    def _loss_func(self, y_hat: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        """Calculate the loss between the output and the input data.

        Parameters
        ----------
        y_hat : torch.Tensor
            The output data from the forward pass.
        y : torch.Tensor
            The input data/label.

        Returns
        -------
        torch.Tensor
            The loss value.
        """
        loss = self.loss_fn(y_hat, y)
        return loss

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Perform a forward pass with the input data on the backbone model.

        Parameters
        ----------
        x : torch.Tensor
            The input data.

        Returns
        -------
        torch.Tensor
            The output data from the forward pass.
        """
        x = self.backbone(x)
        if self.flatten:
            x = x.reshape(x.size(0), -1)
        if self.adapter is not None:
            x = self.adapter(x)
        x = self.fc(x)
        return x

    def _compute_metrics(
        self, y_hat: torch.Tensor, y: torch.Tensor, step_name: str
    ) -> Dict[str, torch.Tensor]:
        """Calculate the metrics for the given step.

        Parameters
        ----------
        y_hat : torch.Tensor
            The output data from the forward pass.
        y : torch.Tensor
            The input data/label.
        step_name : str
            Name of the step. It will be used to get the metrics from the
            `self.metrics` attribute.

        Returns
        -------
        Dict[str, torch.Tensor]
            A dictionary with the metrics values.
        """
        if self.metrics[step_name] is None:
            return {}

        return {
            f"{step_name}_{metric_name}": metric.to(self.device)(y_hat, y)
            for metric_name, metric in self.metrics[step_name].items()
        }

    def _single_step(
        self, batch: torch.Tensor, batch_idx: int, step_name: str
    ) -> torch.Tensor:
        """Perform a single train/validation/test step. It consists in making a
        forward pass with the input data on the backbone model, computing the
        loss between the output and the input data, and logging the loss.

        Parameters
        ----------
        batch : torch.Tensor
            The input data. It must be a 2-element tuple of tensors, where the
            first tensor is the input data and the second tensor is the mask.
        batch_idx : int
            The index of the batch.
        step_name : str
            The name of the step. It will be used to log the loss. The possible
            values are: "train", "val" and "test". The loss will be logged as
            "{step_name}_loss".

        Returns
        -------
        torch.Tensor
            A tensor with the loss value.
        """
        x, y = batch
        y_hat = self.forward(x)
        loss = self._loss_func(y_hat, y)
        self.log(
            f"{step_name}_loss",
            loss,
            on_step=False,
            on_epoch=True,
            prog_bar=True,
            logger=True,
        )

        metrics = self._compute_metrics(y_hat, y, step_name)
        for metric_name, metric_value in metrics.items():
            self.log(
                metric_name,
                metric_value,
                on_step=False,
                on_epoch=True,
                prog_bar=True,
                logger=True,
            )

        if step_name == "train":
            self.train_loss_batches.append(loss)
        elif step_name == "val":
            self.val_loss_batches.append(loss)

        return loss

    def training_step(self, batch: torch.Tensor, batch_idx: int):
        return self._single_step(batch, batch_idx, step_name="train")

    def validation_step(self, batch: torch.Tensor, batch_idx: int):
        return self._single_step(batch, batch_idx, step_name="val")

    def test_step(self, batch: torch.Tensor, batch_idx: int):
        return self._single_step(batch, batch_idx, step_name="test")

    def predict_step(self, batch, batch_idx, dataloader_idx=None):
        x, _ = batch
        y_hat = self.forward(x)
        return y_hat

    def configure_optimizers(self):
        # Freeze or not the backbone model
        for param in self.backbone.parameters():
            param.requires_grad = not self.freeze_backbone
        # Unfreeze the fc model
        #for param in self.fc.parameters():
        #    param.requires_grad = True

        optimizer = self.optimizer(
            self.parameters(), lr=self.learning_rate, **self.optimizer_kwargs
        )
        if self.lr_scheduler is None:
            return optimizer

        scheduler = self.lr_scheduler(optimizer, **self.lr_scheduler_kwargs)

        return [optimizer], [scheduler]

In [6]:
from typing import Any, Dict, Optional, Sequence, Tuple
import os
from collections import OrderedDict
from torch import Tensor, nn, optim
from torchmetrics import Metric
from torchvision.models import ResNet50_Weights
from torchvision.models.resnet import resnet50
from torchvision.models.segmentation.deeplabv3 import ASPP
import torch

class DeepLabV3(SimpleSupervisedModel):
    """A DeeplabV3 with a ResNet50 backbone

    References
    ----------
    Liang-Chieh Chen, George Papandreou, Florian Schroff, Hartwig Adam.
    "Rethinking Atrous Convolution for Semantic Image Segmentation", 2017
    """

    def __init__(
        self,
        backbone: Optional[nn.Module] = None,
        pred_head: Optional[nn.Module] = None,
        loss_fn: Optional[nn.Module] = None,
        learning_rate: float = 0.001,
        num_classes: int = 6,
        pretrained: bool = False,
        weights_path: Optional[str] = None,
        train_metrics: Optional[Dict[str, Metric]] = None,
        val_metrics: Optional[Dict[str, Metric]] = None,
        test_metrics: Optional[Dict[str, Metric]] = None,
        optimizer: type = torch.optim.Adam,
        optimizer_kwargs: Optional[Dict[str, Any]] = None,
        lr_scheduler: Optional[type] = None,
        lr_scheduler_kwargs: Optional[Dict[str, Any]] = None,
        output_shape: Optional[Tuple[int, ...]] = None,
        # add train loss and val loss list
        train_losses = [],
        val_losses = [],
        train_accuracies = [],
        val_accuracies = [],
        train_loss_batches = [],
        val_loss_batches = []
    ):
        """
        Initializes a DeepLabV3 model.

        Parameters
        ----------
        backbone: Optional[nn.Module]
            The backbone network. Defaults to None, which will use a ResNet50
            backbone.
        pred_head: Optional[nn.Module]
            The prediction head network. Defaults to None, which will use a
            DeepLabV3PredictionHead with specified number of classes.
        loss_fn: Optional[nn.Module]
            The loss function. Defaults to None, which will use a
            CrossEntropyLoss.
        learning_rate: float
            The learning rate for the optimizer. Defaults to 0.001.
        num_classes: int
            The number of classes for prediction. Defaults to 6.
        pretrained: bool
            Whether to use pretrained weights. Defaults to False.
        weights_path: Optional[str]
            Path to local pretrained weights file. If provided with pretrained=True,
            loads weights from this path instead of downloading. Defaults to None.
        train_metrics: Optional[Dict[str, Metric]]
            The metrics to be computed during training. Defaults to None.
        val_metrics: Optional[Dict[str, Metric]]
            The metrics to be computed during validation. Defaults to None.
        test_metrics: Optional[Dict[str, Metric]]
            The metrics to be computed during testing. Defaults to None.
        optimizer: type
            Optimizer class to be instantiated. By default, it is set to
            `torch.optim.Adam`. Should be a subclass of
            `torch.optim.Optimizer` (e.g., `torch.optim.SGD`).
        optimizer_kwargs : dict, optional
            Additional kwargs passed to the optimizer constructor.
        lr_scheduler : type, optional
            Learning rate scheduler class to be instantiated. By default, it is
            set to None, which means no scheduler will be used. Should be a
            subclass of `torch.optim.lr_scheduler.LRScheduler` (e.g.,
            `torch.optim.lr_scheduler.StepLR`).
        lr_scheduler_kwargs : dict, optional
            Additional kwargs passed to the scheduler constructor.
        output_shape: Optional[Tuple[int, ...]]
            The output shape of the model. If None, the output shape will be
            the same as the input shape. Defaults to None. This is useful for
            models that require a specific output shape, that is different from
            the input shape.
        """
        backbone = backbone or DeepLabV3Backbone(
            num_classes=num_classes, pretrained=pretrained, weights_path=weights_path
        )
        pred_head = pred_head or DeepLabV3PredictionHead(num_classes=num_classes)
        loss_fn = loss_fn or nn.CrossEntropyLoss()
        self.output_shape = output_shape

        super().__init__(
            backbone=backbone,
            fc=pred_head,
            loss_fn=loss_fn,
            learning_rate=learning_rate,
            train_metrics=train_metrics,
            val_metrics=val_metrics,
            test_metrics=test_metrics,
            optimizer=optimizer,
            optimizer_kwargs=optimizer_kwargs,
            lr_scheduler=lr_scheduler,
            lr_scheduler_kwargs=lr_scheduler_kwargs,
        )

    def forward(self, x: Tensor) -> Tensor:
        """Performs the forward pass of the DeepLabV3 model.

        Parameters
        ----------
        x : Tensor
            Input tensor of shape (batch_size, channels, height, width)

        Returns
        -------
        Tensor
            Output tensor of shape (batch_size, num_classes, height, width)
        """
        x = x.float()
        input_shape = self.output_shape or x.shape[-2:]
        h = self.backbone(x)
        if isinstance(h, OrderedDict):
            h = h["out"]
        z = self.fc(h)
        # Upscaling
        return nn.functional.interpolate(
            z, size=input_shape, mode="bilinear", align_corners=False
        )

    def _loss_func(self, y_hat: Tensor, y: Tensor) -> Tensor:
        """Computes the loss between predictions and ground truth.

        Parameters
        ----------
        y_hat : Tensor
            Predicted tensor of shape (batch_size, num_classes, height, width)
        y : Tensor
            Ground truth tensor of shape (batch_size, 1, height, width)
        """
        return self.loss_fn(y_hat, y.squeeze(1).long())


class DeepLabV3Backbone(nn.Module):
    """A ResNet50 backbone for DeepLabV3"""

    def __init__(
        self,
        num_classes: int = 6,
        pretrained: bool = False,
        weights_path: Optional[str] = None,
    ):
        """
        Initializes the DeepLabV3 backbone model.

        Parameters
        ----------
        num_classes: int
            The number of classes for classification. Default is 6.
        pretrained: bool
            Whether to use pretrained weights. If True and weights_path is None,
            will attempt to download ImageNet pretrained weights. Default is False.
        weights_path: Optional[str]
            Path to local pretrained weights file. If provided with pretrained=True,
            loads weights from this path instead of downloading. Default is None.
        """
        super().__init__()

        if pretrained and weights_path is not None:
            # Validate file path exists
            if not os.path.exists(weights_path):
                raise FileNotFoundError(f"Weights file not found: {weights_path}")

            # Load from local weights file
            RN50model = resnet50(replace_stride_with_dilation=[False, True, True])

            state_dict = torch.load(weights_path, map_location="cpu")
            # Handle different weight file formats
            if "state_dict" in state_dict:
                state_dict = state_dict["state_dict"]
            elif "model" in state_dict:
                state_dict = state_dict["model"]

            # Filter out classifier weights if they exist (fc layer)
            # since we don't use them in the backbone
            filtered_state_dict = {
                k: v for k, v in state_dict.items() if not k.startswith("fc.")
            }

            # Load the filtered state dict
            missing_keys, unexpected_keys = RN50model.load_state_dict(
                filtered_state_dict, strict=False
            )

            if missing_keys:
                print(
                    f"Warning: Missing keys when loading pretrained weights: {missing_keys}"
                )
            if unexpected_keys:
                print(
                    f"Warning: Unexpected keys when loading pretrained weights: {unexpected_keys}"
                )

            print(f"Successfully loaded pretrained weights from {weights_path}")

        elif pretrained and weights_path is None:
            # Use torchvision's pretrained weights (requires internet)
            RN50model = resnet50(
                weights=ResNet50_Weights.IMAGENET1K_V1,
                replace_stride_with_dilation=[False, True, True],
            )
            print("Successfully loaded ImageNet pretrained weights from torchvision")
        else:
            # No pretrained weights, random initialization
            RN50model = resnet50(replace_stride_with_dilation=[False, True, True])

        self.RN50model = RN50model

    def freeze_weights(self):
        """Freezes all parameters in the backbone, making them non-trainable."""
        for param in self.RN50model.parameters():
            param.requires_grad = False

    def unfreeze_weights(self):
        """Unfreezes all parameters in the backbone, making them trainable."""
        for param in self.RN50model.parameters():
            param.requires_grad = True

    def forward(self, x):
        """Performs the forward pass of the backbone.

        Parameters
        ----------
        x : Tensor
            Input tensor of shape (batch_size, channels, height, width)

        Returns
        -------
        Tensor
            Feature map tensor from the ResNet50 backbone
        """
        x = self.RN50model.conv1(x)
        x = self.RN50model.bn1(x)
        x = self.RN50model.relu(x)
        x = self.RN50model.maxpool(x)
        x = self.RN50model.layer1(x)
        x = self.RN50model.layer2(x)
        x = self.RN50model.layer3(x)
        x = self.RN50model.layer4(x)
        # x = self.RN50model.avgpool(x)      # These should be removed for deeplabv3
        # x = torch.RN50model.flatten(x, 1)  # These should be removed for deeplabv3
        # x = self.RN50model.fc(x)           # These should be removed for deeplabv3
        return x


class DeepLabV3PredictionHead(nn.Sequential):
    """The prediction head for DeepLabV3"""

    def __init__(
        self,
        in_channels: int = 2048,
        num_classes: int = 6,
        atrous_rates: Sequence[int] = (12, 24, 36),
    ) -> None:
        """
        Initializes the DeepLabV3 prediction head.

        Parameters
        ----------
        in_channels: int
            Number of input channels. Defaults to 2048.
        num_classes: int
            Number of output classes. Defaults to 6.
        atrous_rates: Sequence[int]
            A sequence of atrous rates for the ASPP module. Defaults to (12, 24, 36).
        """
        super().__init__(
            ASPP(in_channels, atrous_rates),
            nn.Conv2d(256, 256, 3, padding=1, bias=False),
            #nn.Conv2d(256, 256, 1, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Conv2d(256, num_classes, 1),
        )

    def forward(self, input) -> Tensor:
        """Performs the forward pass of the prediction head.

        Parameters
        ----------
        input : Tensor
            Input tensor from the backbone

        Returns
        -------
        Tensor
            Output tensor with class predictions
        """
        #assert input.shape[0] > 1, "Batch size must be greater than 1 due to BatchNorm"
        return super().forward(input)

# functions

In [7]:
class SimpleSegmentationHead(nn.Sequential):
    def __init__(self, in_channels, num_classes):
        super().__init__(
            nn.Conv2d(in_channels, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, num_classes, kernel_size=1)
        )

In [8]:
class Identity_2(_Transform):
    """This class is a dummy transform that does nothing. It is useful when
    you want to skip a transform in a pipeline.
    """

    def __call__(self, x: np.ndarray) -> np.ndarray:
        normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        transform = transforms.Compose([transforms.ToTensor(), normalize])
        return transform(x)

    def __str__(self) -> str:
        return "Identity()"

In [9]:
class Format_label_img(_Transform):

    def __call__(self, x: np.ndarray) -> torch.Tensor:

        if x.ndim == 3:
            x = x.squeeze()

        label_tensor = torch.from_numpy(x).long()

        if label_tensor.min() >= 1:
            label_tensor = label_tensor - 1

        return label_tensor

In [10]:
def model_mIoU(model, trainer, dataloader):
    predictions = trainer.predict(model, dataloader)
    predictions = torch.cat(predictions, dim=0)
    pred_classes = torch.argmax(predictions, dim=1)
    test_samples = np.array([y for _, y in dataloader.test_dataset])
    y = torch.from_numpy(test_samples).long()
    jaccard = JaccardIndex(task="multiclass", num_classes=3)
    score = jaccard(pred_classes, y)
    #print(f"The mIoU of the model is {score.item()*100:.2f}%")
    return score.item()*100

In [13]:
def frag_data_scores(backbone, name):

    with open(DIR + f"frag_dataset_results/frag_data_lr_{name}.csv", mode="w", newline="") as file:
            writer = csv.writer(file)
            writer.writerow(["number_data", "mIoU"])

    # each number corresponds to a folder in the fragmented dataset, with each folder containing 0.2%, 1%, and 10% of the training dataset, respectively
    number_data = ["01","06","61"]

    for i in number_data:

        _backbone = copy.deepcopy(backbone)
        model = DeepLabV3(_backbone,SimpleSegmentationHead(2048,3),learning_rate=1e-4,num_classes=3)

        # path to fragmented dataset
        train_fine_data_reader = PNGReader(
            path=Path(DIR + f'dataset_spectogram/train/fragmented_data_2/iteration_{i}/data/')
        )
        train_fine_labels_reader = PNGReader(
            path=Path(DIR + f'dataset_spectogram/train/fragmented_data_2/iteration_{i}/label/')
        )

        train_fine_dataset = SimpleDataset(
            readers=[train_fine_data_reader, train_fine_labels_reader],
            transforms=[Identity_2(), Format_label_img()],
        )

        data_fine_module = MinervaDataModule(
            train_dataset=train_fine_dataset,
            val_dataset=val_fine_dataset,
            test_dataset=test_fine_dataset,
            batch_size=32,
            num_workers=8,
            name="simulated spectogram"
        )

        trainer_base = L.Trainer(
        max_epochs=50,
        accelerator="gpu",
        devices=1,
        logger=False,
        enable_checkpointing=False,
        )

        trainer_base.fit(model, data_fine_module)

        miou_score = model_mIoU(model, _trainer, data_fine_module)

        with open(DIR + f"frag_dataset_results/frag_data_lr_{name}.csv", mode="a", newline="") as file:
            writer = csv.writer(file)
            writer.writerow([i, miou_score])

        del(model)

In [14]:
def frag_data_scores_2(backbone, name):

    with open(DIR + f"frag_dataset_results/frag_data_lr_{name}_2.csv", mode="w", newline="") as file:
            writer = csv.writer(file)
            writer.writerow(["percent", "mIoU_list", "mIoU_mean"])

    frag = 50
    for i in range(2):
        miou_list = []
        if i < 1:
            for j in range(2):

                _backbone = copy.deepcopy(backbone)
                model = DeepLabV3(_backbone,SimpleSegmentationHead(2048,3),learning_rate=1e-3,num_classes=3)

                item = str(frag)

                # path to fragmented dataset
                train_fine_data_reader = PNGReader(
                    path=Path(DIR + f'dataset_spectogram/train/fragmented_data/{item}_percent/data/')
                )
                train_fine_labels_reader = PNGReader(
                    path=Path(DIR + f'dataset_spectogram/train/fragmented_data/{item}_percent/label/')
                )

                train_fine_dataset = SimpleDataset(
                    readers=[train_fine_data_reader, train_fine_labels_reader],
                    transforms=[Identity_2(), Format_label_img()],
                )

                data_fine_module = MinervaDataModule(
                    train_dataset=train_fine_dataset,
                    val_dataset=val_fine_dataset,
                    test_dataset=test_fine_dataset,
                    batch_size=32,
                    num_workers=8,
                    #drop_last=True,
                    name="simulated spectogram"
                )

                trainer_base = L.Trainer(
                max_epochs=50,
                accelerator="gpu",
                devices=1,
                logger=False,
                enable_checkpointing=False,
                )

                trainer_base.fit(model, data_fine_module)

                miou_score = model_mIoU(model, _trainer, data_fine_module)
                miou_list.append(miou_score)
                del(model)

        else:
            for p in range(5):

                _backbone = copy.deepcopy(backbone)
                model = DeepLabV3(_backbone,SimpleSegmentationHead(2048,3),learning_rate=1e-3,num_classes=3)

                train_fine_data_reader = PNGReader(
                path=Path(DIR + 'dataset_spectogram/train/data')
                )
                train_fine_labels_reader = PNGReader(
                    path=Path(DIR + 'dataset_spectogram/train/label')
                )

                train_fine_dataset = SimpleDataset(
                    readers=[train_fine_data_reader, train_fine_labels_reader],
                    transforms=[Identity_2(), Format_label_img()],
                )

                data_fine_module = MinervaDataModule(
                    train_dataset=train_fine_dataset,
                    val_dataset=val_fine_dataset,
                    test_dataset=test_fine_dataset,
                    batch_size=32,
                    num_workers=8,
                    additional_train_dataloader_kwargs=False,
                    name="simulated spectogram"
                )

                trainer_base = L.Trainer(
                    max_epochs=50,
                    accelerator="gpu",
                    devices=1,
                    logger=False,
                    enable_checkpointing=False,
                    #callbacks=[checkpoint_callback]
                )

                trainer_base.fit(model, data_fine_module)

                miou_score = model_mIoU(model, _trainer, data_fine_module)

                miou_list.append(miou_score)
                del(model)

        with open(DIR + f"frag_dataset_results/frag_data_lr_{name}_2.csv", mode="a", newline="") as file:
            writer = csv.writer(file)
            writer.writerow([frag, miou_list, sum(miou_list)/len(miou_list)])

        frag += 25

# model config

In [15]:
val_fine_data_reader = PNGReader(
    path=Path(DIR + 'dataset_spectogram/val/data/')
)

In [16]:
val_fine_labels_reader = PNGReader(
    path=Path(DIR + 'dataset_spectogram/val/label')
)

In [17]:
test_fine_data_reader = PNGReader(
    path=Path(DIR + 'dataset_spectogram/test/data')
)

In [18]:
test_fine_labels_reader = PNGReader(
    path=Path(DIR + 'dataset_spectogram/test/label')
)

In [19]:
val_fine_dataset = SimpleDataset(
    readers=[val_fine_data_reader, val_fine_labels_reader],
    transforms=[Identity_2(), Format_label_img()],
)

print(val_fine_dataset)

           📂 SimpleDataset Information            
📌 Dataset Type: SimpleDataset
   └── Reader 0: PNGReader at '/content/drive/My Drive/iniciacao_cientifica/dataset_spectogram/val/data' (525 files)
   │     └── Transform: Identity()
   └── Reader 1: PNGReader at '/content/drive/My Drive/iniciacao_cientifica/dataset_spectogram/val/label' (525 files)
   │     └── Transform: <__main__.Format_label_img object at 0x7eec9e7b1160>
   │
   └── Total Readers: 2


In [20]:
test_fine_dataset = SimpleDataset(
    readers=[test_fine_data_reader, test_fine_labels_reader],
    transforms=[Identity_2(), Format_label_img()],
)

print(test_fine_dataset)

           📂 SimpleDataset Information            
📌 Dataset Type: SimpleDataset
   └── Reader 0: PNGReader at '/content/drive/My Drive/iniciacao_cientifica/dataset_spectogram/test/data' (266 files)
   │     └── Transform: Identity()
   └── Reader 1: PNGReader at '/content/drive/My Drive/iniciacao_cientifica/dataset_spectogram/test/label' (266 files)
   │     └── Transform: <__main__.Format_label_img object at 0x7eec9e7b3c20>
   │
   └── Total Readers: 2


In [ ]:
RN50model_default_dilatation = resnet50(weights="DEFAULT",replace_stride_with_dilation=[False, False, True])
RN50model_none_dilatation = resnet50(weights=None,replace_stride_with_dilation=[False, False, True])

# Base

In [23]:
resnet50_backbone_base = torch.nn.Sequential(*list(RN50model_none_dilatation.children())[:-2])

In [ ]:
frag_data_scores(resnet50_backbone_base, "base")

In [ ]:
frag_data_scores_2(resnet50_backbone_base, "base")

# ImageNet

In [ ]:
resnet50_backbone_imagenet = torch.nn.Sequential(*list(RN50model_default_dilatation.children())[:-2])

In [ ]:
frag_data_scores(resnet50_backbone_imagenet, "imagenet")

In [ ]:
frag_data_scores_2(resnet50_backbone_imagenet, "imagenet")

# CoCo

In [ ]:
model = deeplabv3_resnet50(weights="COCO_WITH_VOC_LABELS_V1")
backbone_coco = model.backbone

In [ ]:
frag_data_scores(backbone_coco, "CoCo")

In [ ]:
frag_data_scores_2(backbone_coco, "CoCo")

# BYOL

In [ ]:
# path to BYOL checkpoint
byol_config = BYOL.load_from_checkpoint(DIR + "checkpoints/byol_config_1_powder_epochepoch=29.ckpt", backbone=torch.nn.Sequential(*list(RN50model_default_dilatation.children())[:-1]), strict=True)
_backbone_config = byol_config.backbone

In [ ]:
deeplab_backbone = torch.nn.Sequential(*list(_backbone_config.children())[:-1])

In [ ]:
frag_data_scores(deeplab_backbone, "config_1")

In [ ]:
frag_data_scores_2(deeplab_backbone_rotatecrop_3, "config_1")